# ShieldML — Exploratory Data Analysis & Anomaly Detection

This notebook conducts a deep-dive statistical exploration of the **ShieldML financial transactions dataset** to uncover fraud typologies, analyze class imbalance, verify temporal distributions, and establish the feature engineering strategy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
df = pd.read_csv("../data/raw/transactions.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Loaded {len(df):,} transactions spanning {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

### 1. Class Distribution & Imbalance Audit

Financial fraud is inherently rare. Below we evaluate the empirical class ratio to determine necessary imbalance mitigation strategies (e.g., SMOTE, ADASYN, and cost-sensitive scale weights).

In [ ]:
counts = df["is_fraud"].value_counts()
fraud_rate = df["is_fraud"].mean() * 100

print(f"Legitimate Transactions (0): {counts[0]:,} ({100 - fraud_rate:.2f}%)")
print(f"Fraudulent Transactions (1): {counts[1]:,} ({fraud_rate:.2f}%)")
print(f"Imbalance Ratio: 1:{int(counts[0] / counts[1])}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="is_fraud", palette=["#10B981", "#EF4444"], ax=ax)
ax.set_xticklabels(["Legitimate (0)", "Fraud (1)"])
ax.set_title("Transaction Class Distribution (Severe Imbalance)", fontsize=12, fontweight="bold")
plt.show()

### 2. Diurnal & Temporal Circadian Patterns

Legitimate transactions correlate with human waking cycles, while automated fraud scripts and account takeovers frequently cluster during nocturnal windows (00:00 to 05:00).

In [ ]:
df["hour"] = df["timestamp"].dt.hour
hourly_stats = df.groupby("hour")["is_fraud"].agg(["count", "mean"]).reset_index()
hourly_stats["fraud_pct"] = hourly_stats["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=hourly_stats, x="hour", y="fraud_pct", marker="o", color="#EF4444", lw=2.5, ax=ax)
ax.set_xticks(range(0, 24))
ax.set_xlabel("Hour of Day (0-23)", fontweight="bold")
ax.set_ylabel("Fraud Rate (%)", fontweight="bold")
ax.set_title("Diurnal Fraud Prevalence by Hour of Day", fontsize=13, fontweight="bold")
ax.axvspan(0, 5, color="#FEE2E2", alpha=0.5, label="High-Risk Nocturnal Window")
ax.legend()
plt.show()

### 3. Transaction Amount & Merchant Z-Score Analysis

Comparing transaction amounts across merchant categories reveals that fraud often manifests as abnormal spikes above historical merchant baselines.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x="merchant_category", y="amount", hue="is_fraud", palette=["#10B981", "#EF4444"], ax=ax, showfliers=False)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_title("Transaction Amount Distribution across Merchant Categories by Class", fontsize=13, fontweight="bold")
plt.show()

### 4. Conclusions & Modeling Strategy

1. **Severe Imbalance:** Class 1 represents ~2.2% of the dataset. Models must utilize cost-sensitive weighting (`scale_pos_weight`) and SMOTE oversampling exclusively within training partitions.
2. **Feature Engineering Targets:**
   - Sine and cosine cyclical encodings of `hour` and `dayofweek`.
   - Backward-looking 1-hour, 24-hour, and 7-day velocity aggregations.
   - Merchant-specific amount z-scores.
   - Novel device flags.
3. **Threshold Calibration:** A standard 0.50 cutoff misses low-probability early fraud signals. Maximizing $F_2$ on validation data will prioritize recall and minimize false negatives.